# Seasonal Agriculture Performance Analysis

VOIS AICTE Major Project - Data Analytics

Dataset: 4000 farm records, 3 cropping seasons (Kharif / Rabi / Zaid), 28 columns.

Objective: quantify how yield, cost, revenue, profit, water use and disease risk
differ by season, and check whether the differences are consistent enough to act on
(not just an artifact of a few states or a few crops).

Sections: preprocessing -> feature engineering -> seasonal EDA -> significance test -> summary.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.rcParams['figure.figsize'] = (9, 5)
plt.rcParams['axes.titlesize'] = 12
pd.set_option('display.max_columns', 30)

# custom palette instead of a seaborn default - roughly monsoon green / winter gold / summer rust,
# matches the three seasons in order so every chart below stays visually consistent
SEASON_ORDER = ['Kharif', 'Rabi', 'Zaid']
SEASON_COLORS = ['#4C7A3D', '#C99A3E', '#A8452F']


## Data Preprocessing

In [ ]:
df = pd.read_csv('seasonal_agriculture_performance_dataset.csv')
print(df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe().T
# nothing alarming here - no negative costs/revenue, ranges look agronomically sane


In [ ]:
missing = df.isnull().sum()
missing[missing > 0]


In [ ]:
print('duplicate rows:', df.duplicated().sum())
print('duplicate Farm_IDs:', df['Farm_ID'].duplicated().sum())


Missing values are confined to `Rainfall_mm`, `Soil_Moisture_pct`, `Yield_Tonnes_Ha` (~1% each).
Filling with the overall column mean would wash out the seasonal signal this whole
project is built around, so imputing per-season median instead.

In [ ]:
cols_to_fill = ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']
for col in cols_to_fill:
    df[col] = df.groupby('Season')[col].transform(lambda s: s.fillna(s.median()))

df[cols_to_fill].isnull().sum()


Outlier check (IQR, 1.5x rule) - flagging only, not dropping. A crop failure or a
bumper Sugarcane yield is a real seasonal signal, not a data entry error, so removing
extreme values here would defeat the purpose of the analysis.

In [ ]:
def count_outliers_iqr(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lo) | (series > hi)).sum()

numeric_cols = df.select_dtypes(include=np.number).columns
outlier_counts = df[numeric_cols].apply(count_outliers_iqr).sort_values(ascending=False)
outlier_counts[outlier_counts > 0]


Counts are small relative to 4000 rows - kept as-is. (Sanity-checked a handful of the
Water_Efficiency outliers manually; several sit at suspiciously round numbers like
50.0 exactly, which smells like a simulated/capped value in the source data rather
than a real farm. Doesn't change the seasonal comparison either way, so leaving it.)

In [ ]:
for col in ['State', 'District', 'Crop', 'Season', 'Irrigation_Method']:
    df[col] = df[col].str.strip().str.title()

df[['State', 'District', 'Crop', 'Season', 'Irrigation_Method']].nunique()


## Feature Engineering

In [ ]:
# profit margin - normalises profit by revenue so farms of very different scale are comparable
df['Profit_Margin_pct'] = df['Profit_INR'] / df['Revenue_INR'] * 100

# cost per tonne produced - a rough efficiency metric
df['Cost_per_Tonne_INR'] = df['Total_Cost_INR'] / df['Production_Tonnes']

# used later for the loss-rate comparison across seasons
df['Is_Loss_Making'] = df['Profit_INR'] < 0

df[['Profit_Margin_pct', 'Cost_per_Tonne_INR', 'Is_Loss_Making']].describe(include='all')


## Exploratory Data Analysis

### Season Distribution

In [ ]:
season_counts = df['Season'].value_counts().reindex(SEASON_ORDER)
print(season_counts)

plt.figure(figsize=(6, 4))
sns.barplot(x=season_counts.index, y=season_counts.values, palette=SEASON_COLORS)
plt.title('Farm Records per Season')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


Kharif ~44%, Rabi ~41%, Zaid ~15%. Zaid being the smallest sample tracks with it being
the short summer window squeezed between the two main seasons - keep this in mind before
over-reading any Zaid-specific numbers below.

### Core Metrics by Season

In [ ]:
season_summary = df.groupby('Season')[
    ['Yield_Tonnes_Ha', 'Total_Cost_INR', 'Revenue_INR', 'Profit_INR', 'Profit_Margin_pct']
].mean().round(2).reindex(SEASON_ORDER)
season_summary


In [ ]:
# reusable so the 4 panels below don't turn into 4 near-identical copy-pasted blocks
def season_bar(ax, col, title, ylabel, zero_line=False):
    sns.barplot(data=df, x='Season', y=col, estimator=np.mean, order=SEASON_ORDER,
                palette=SEASON_COLORS, ax=ax)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel('')
    if zero_line:
        ax.axhline(0, color='black', linewidth=1, linestyle='--')

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
season_bar(axes[0, 0], 'Yield_Tonnes_Ha', 'Avg Yield', 'Tonnes/Ha')
season_bar(axes[0, 1], 'Profit_INR', 'Avg Profit', 'INR', zero_line=True)
season_bar(axes[1, 0], 'Total_Cost_INR', 'Avg Cost', 'INR')
season_bar(axes[1, 1], 'Revenue_INR', 'Avg Revenue', 'INR')
plt.tight_layout()
plt.show()


Kharif leads on yield and profit. Zaid profit is negative on average despite cost
being roughly flat across seasons - the shortfall is on the revenue side, not spend.

In [ ]:
loss_rate = df.groupby('Season')['Is_Loss_Making'].mean().mul(100).round(1).reindex(SEASON_ORDER)
print(loss_rate)

plt.figure(figsize=(6, 4))
sns.barplot(x=loss_rate.index, y=loss_rate.values, palette=SEASON_COLORS)
plt.ylabel('% farms with negative profit')
plt.title('Loss-Making Farms by Season')
plt.tight_layout()
plt.show()


~65% of Zaid farms run at a loss vs under half in Kharif - probably the clearest single
finding in this dataset.

### Environmental Conditions

In [ ]:
env_colors = ['#3B6EA5', '#C9603C', '#4C7A3D']  # rainfall / temp / humidity

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, color in zip(axes, ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct'], env_colors):
    sns.boxplot(data=df, x='Season', y=col, order=SEASON_ORDER, color=color, ax=ax)
    ax.set_title(col.replace('_', ' '))
plt.tight_layout()
plt.show()


Zaid = hottest, driest of the three. Consistent with it being the irrigation-dependent
summer season - ties into the profit gap above, see irrigation efficiency section.

### Crop x Season Yield

In [ ]:
crop_season_yield = df.pivot_table(index='Crop', columns='Season',
                                    values='Yield_Tonnes_Ha', aggfunc='mean').round(2)
crop_season_yield = crop_season_yield[SEASON_ORDER]
crop_season_yield


In [ ]:
from matplotlib.colors import LinearSegmentedColormap
earth_cmap = LinearSegmentedColormap.from_list('earth', ['#F5F0E1', '#8CA86E', '#2F5233'])

plt.figure(figsize=(10, 6))
sns.heatmap(crop_season_yield, annot=True, fmt='.2f', cmap=earth_cmap, linewidths=0.5)
plt.title('Avg Yield (t/ha) - Crop x Season')
plt.tight_layout()
plt.show()


Sugarcane sits on a totally different scale (tens of t/ha vs low single digits for
everything else) - agronomically normal, but it flattens the color scale for every
other crop, so re-plotting without it below.

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(crop_season_yield.drop('Sugarcane'), annot=True, fmt='.2f',
            cmap=earth_cmap, linewidths=0.5)
plt.title('Avg Yield (t/ha) - Crop x Season, Sugarcane excluded')
plt.tight_layout()
plt.show()


Rice/Maize peak in Kharif, Wheat peaks in Rabi - matches standard sowing calendars
(wheat = winter crop, rice/maize = monsoon crops). Pulses trail in every season.

### Irrigation Efficiency

In [ ]:
irrigation_colors = {'Drip': '#2F5233', 'Rainfed': '#6B8F47', 'Sprinkler': '#C99A3E', 'Flood': '#A8452F'}

irrigation_eff = (df.groupby('Irrigation_Method')['Water_Efficiency_t_per_1000m3']
                    .mean().sort_values(ascending=False).round(2))
print(irrigation_eff)

plt.figure(figsize=(7, 4.5))
sns.barplot(x=irrigation_eff.index, y=irrigation_eff.values,
            palette=[irrigation_colors.get(m, '#888888') for m in irrigation_eff.index])
plt.title('Water Efficiency by Irrigation Method')
plt.ylabel('t produced per 1000 m3')
plt.tight_layout()
plt.show()


In [ ]:
# is Zaid over-reliant on the less efficient methods?
irrigation_mix = pd.crosstab(df['Season'], df['Irrigation_Method'], normalize='index').mul(100).round(1)
irrigation_mix = irrigation_mix.reindex(SEASON_ORDER)
irrigation_mix


In [ ]:
irrigation_mix[list(irrigation_colors.keys())].plot(
    kind='bar', stacked=True, figsize=(8, 5),
    color=[irrigation_colors[m] for m in irrigation_colors])
plt.title('Irrigation Method Mix by Season')
plt.ylabel('% of farms')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='Method')
plt.tight_layout()
plt.show()


Drip/Rainfed = most efficient, Flood = worst. Zaid leaning on Flood irrigation while
also getting the least rainfall of the three seasons lines up directly with its weaker
profit numbers - this is probably the single most actionable finding here.

### Disease / Pest Risk

In [ ]:
plt.figure(figsize=(7, 5))
sns.violinplot(data=df, x='Season', y='Disease_Pest_Risk_pct', order=SEASON_ORDER,
               palette=SEASON_COLORS)
plt.title('Disease/Pest Risk by Season')
plt.tight_layout()
plt.show()


Kharif has the highest average risk and widest spread - monsoon humidity favours
pests/disease, as expected. Rabi and Zaid are drier and lower-risk.

### Correlation with Yield

In [ ]:
key_cols = ['Yield_Tonnes_Ha', 'Rainfall_mm', 'Avg_Temperature_C', 'Soil_Moisture_pct',
            'Nitrogen_kg_ha', 'Fertilizer_kg_ha', 'Water_Efficiency_t_per_1000m3',
            'Disease_Pest_Risk_pct', 'Profit_INR', 'Market_Price_INR_Tonne']

corr_matrix = df[key_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Matrix - Key Variables')
plt.tight_layout()
plt.show()


`Water_Efficiency_t_per_1000m3` correlates with yield more strongly than fertiliser,
nitrogen or rainfall individually - wasn't expecting it to dominate the others by this
much. `Market_Price_INR_Tonne` correlates negatively with yield, which is just
supply and demand (bumper output -> lower price per tonne) rather than anything wrong
with the data, but worth stating explicitly since it's the non-obvious result.

### State-Wise View

In [ ]:
state_profit = df.groupby('State')['Profit_INR'].mean().sort_values(ascending=False).round(0)

# color negative-average states differently - makes the loss-making states pop out immediately
bar_colors = ['#2F5233' if v > 0 else '#A8452F' for v in state_profit.values]

plt.figure(figsize=(9, 5))
sns.barplot(x=state_profit.values, y=state_profit.index, palette=bar_colors)
plt.title('Avg Profit by State (all seasons combined)')
plt.xlabel('INR')
plt.tight_layout()
plt.show()


In [ ]:
state_season_profit = df.pivot_table(index='State', columns='Season',
                                      values='Profit_INR', aggfunc='mean').round(0)
state_season_profit = state_season_profit[SEASON_ORDER]
state_season_profit


Punjab and Maharashtra top the ranking, Andhra Pradesh lowest. The Kharif > Rabi > Zaid
ordering from earlier holds inside most individual states too, so it looks like a real
seasonal effect rather than one or two states dragging the average around.

## Statistical Significance Check

In [ ]:
kharif_p = df.loc[df['Season'] == 'Kharif', 'Profit_INR']
rabi_p   = df.loc[df['Season'] == 'Rabi',   'Profit_INR']
zaid_p   = df.loc[df['Season'] == 'Zaid',   'Profit_INR']

f_stat, p_value = stats.f_oneway(kharif_p, rabi_p, zaid_p)
print(f'F = {f_stat:.2f}, p = {p_value:.6f}')

# standard ANOVA assumes roughly equal variance across groups - not formally tested here
# (e.g. Levene's test), so treat the p-value as indicative rather than exact
print('significant at 0.05' if p_value < 0.05 else 'not significant at 0.05')


## Key Insights

1. Kharif is the strongest season - highest yield (~5.6 t/ha) and the only season with
   positive average profit (~Rs 1.79 lakh/farm).
2. Zaid is weakest financially - negative average profit, ~65% of farms loss-making,
   despite cost staying roughly flat across seasons. Shortfall is on revenue, not spend.
3. Zaid is also the hottest/driest season and leans more on Flood irrigation, the least
   efficient method - water efficiency is the strongest single driver of yield in the
   dataset, ahead of fertiliser and nitrogen.
4. Crop choice is season-dependent: Rice/Maize best in Kharif, Wheat best in Rabi,
   Pulses weakest everywhere.
5. Disease/pest risk peaks in Kharif (monsoon humidity) but doesn't appear to be
   eroding Kharif's profit lead at current levels.
6. Market price moves inversely with yield - a supply effect that partly offsets the
   benefit of a higher-yield season/crop.
7. The Kharif > Rabi > Zaid profit ordering holds across most states, and the ANOVA
   result supports it being a real seasonal effect rather than noise.

**Recommendations**
- Shift Zaid irrigation mix toward Drip/Rainfed, away from Flood.
- Re-evaluate Zaid crop selection - may suit drought-tolerant crops better than the
  current mix.
- Kharif's pest/disease risk is already elevated; targeted pest management could
  protect margins as risk increases.
- Don't assume higher yield = higher income automatically - the price-yield tradeoff
  means market access/timing matters too.

**Limitations**
- Dataset is a single-year snapshot (assumed) - no multi-year trend to confirm these
  patterns hold beyond this sample.
- ANOVA assumes equal variance across season groups; not separately verified here.
- Some fields (e.g. a subset of Water_Efficiency values) show suspiciously round
  numbers, possibly simulated/capped in the source data - didn't materially affect
  direction of the seasonal comparisons, but worth flagging for anyone extending this.